In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
import matplotlib.pyplot as plt
from tqdm import tqdm
from dataset import prepare_sampled_data, CrackDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device : {device}")

device : cuda


In [5]:
# data path
BASE_DIR = r"D:\Study\HumanStudy\Dataset\Training"

# image preprocessing
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# data loader
train_data_list = prepare_sampled_data(BASE_DIR)
train_dataset = CrackDataset(train_data_list, transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)

[D:\Study\HumanStudy\Dataset\Training]


  0%|          | 0/419995 [00:00<?, ?it/s]



 전체 데이터 수 : 138,398장
 정상(0)     : 20,000장
 단순 균열(1): 40,000장 
 기타 결함(1): 78,398장


In [10]:
model_name = 'efficientnet_b0'
model = timm.create_model(model_name, pretrained=True, num_classes=2)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# train history
history = {'loss': [], 'acc': []}
num_epochs = 10
best_loss = float('inf')

In [16]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    corrects = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]")
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        corrects += torch.sum(preds == labels.data)
        total += labels.size(0)
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    scheduler.step()
    epoch_loss = running_loss / total
    epoch_acc = corrects.item() / total
    
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    print(f"Epoch {epoch+1} : 평균 Loss: {epoch_loss:.4f}, Accuracy : {epoch_acc:.4f}")

    # best weight
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), f"best_{model_name}_crack_classifier.pth")
        print(f" best model Loss: {best_loss:.4f} ")

Epoch [1/10]: 100%|███████████████████████████████████████████████████| 2163/2163 [25:59<00:00,  1.39it/s, loss=0.0076]


Epoch 1 : 평균 Loss: 0.0288, Accuracy : 0.9906
 best model Loss: 0.0288 


Epoch [2/10]: 100%|███████████████████████████████████████████████████| 2163/2163 [26:56<00:00,  1.34it/s, loss=0.0003]


Epoch 2 : 평균 Loss: 0.0228, Accuracy : 0.9926
 best model Loss: 0.0228 


Epoch [3/10]: 100%|███████████████████████████████████████████████████| 2163/2163 [26:51<00:00,  1.34it/s, loss=0.0008]


...

Epoch 8 : 평균 Loss: 0.0018, Accuracy : 0.9994
 best model Loss: 0.0018 


Epoch [9/10]: 100%|███████████████████████████████████████████████████| 2163/2163 [22:58<00:00,  1.57it/s, loss=0.0000]


Epoch 9 : 평균 Loss: 0.0011, Accuracy : 0.9997
 best model Loss: 0.0011 


Epoch [10/10]: 100%|██████████████████████████████████████████████████| 2163/2163 [23:58<00:00,  1.50it/s, loss=0.0000]

Epoch 10 : 평균 Loss: 0.0003, Accuracy : 0.9999
 best model Loss: 0.0003 
